# 05: Machine learning (Phase 5)

**Goal:** predict whether a company will be distressed next year (Altman Z' < 1.23 in year t+1) from its financial ratios in year t, compare four models against two simple baselines, and explain the best model.

**Input:** `data/processed/modelling_table.csv` (Phase 4).

**Design decisions (approved):**
- Companies with revenue below USD 10 million are excluded from modelling (checked in a sensitivity run); 18 features.
- Time-based split by feature year: train 2015-2020, validation 2021-2022, test 2023-2024.
- Preprocessing fitted on training rows only: winsorising at the 1st/99th percentile, median imputation of blank ratios with missing-value indicators, scaling for logistic regression. XGBoost handles blanks itself.
- Class weights for the imbalance; settings chosen on validation PR-AUC; threshold chosen on validation (best F1); final models refitted on train + validation and evaluated on test once.
- Baselines: current ratio < 1, and persistence (distressed next year if Z' < 1.23 this year).
- Case-company probabilities by walk-forward: each year scored by a model trained only on earlier years.

**Outputs** (`data/processed/`): `model_comparison.csv`, `tuning_results.csv`, `test_predictions.csv`, `feature_importance.csv`, `walk_forward_predictions.csv`, `robustness_checks.csv`.

In [1]:
import sys
import warnings
import numpy as np
import pandas as pd
import shap
from sklearn.inspection import permutation_importance

sys.path.insert(0, "..")
from src import config
from src import models as M

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:,.3f}".format)

SEED = config.RANDOM_SEED
np.random.seed(SEED)
print("Random seed:", SEED)

mt = pd.read_csv(config.PROCESSED_DIR / "modelling_table.csv")
zs = pd.read_csv(config.PROCESSED_DIR / "zscores.csv")[["cik", "label_year", "z_prime", "distressed"]]
df = M.prepare(mt).merge(zs, on=["cik", "label_year"], how="left")
masks = M.split(df, config.TRAIN_YEARS, config.VAL_YEARS, config.TEST_YEARS)
train, val, test = (df[masks[k]] for k in ("train", "val", "test"))
train_val = df[masks["train"] | masks["val"]]

print(f"Rows after excluding low-revenue companies: {len(df):,} ({df[M.TARGET].notna().sum():,} labelled)")
print(f"Features ({len(M.FEATURES)}): {M.FEATURES}")
pd.DataFrame({k: {"feature years": f"{y[0]}-{y[1]}", "rows": len(d), "distressed": int(d[M.TARGET].sum()),
                  "share": d[M.TARGET].mean()}
              for k, y, d in [("train", config.TRAIN_YEARS, train), ("validation", config.VAL_YEARS, val),
                              ("test", config.TEST_YEARS, test)]}).T

Random seed: 42
Rows after excluding low-revenue companies: 819 (670 labelled)
Features (18): ['current_ratio', 'quick_ratio', 'cash_ratio', 'debt_to_equity', 'debt_ratio', 'interest_coverage', 'gross_margin', 'operating_margin', 'net_margin', 'roa', 'roe', 'asset_turnover', 'revenue_growth', 'net_income_growth', 'log_total_assets', 'negative_equity', 'no_interest_expense', 'coverage_uses_proxy']


,feature years,rows,distressed,share
train,2015-2020,397,152,0.383
validation,2021-2022,139,43,0.309
test,2023-2024,134,54,0.403


## Step 1: Tune each model on the validation years

Every setting in a small grid is fitted on the training years and scored on validation by PR-AUC (area under the precision-recall curve), which focuses on how well distressed companies are ranked. The decision threshold is then the probability cutoff with the best F1 for the distressed class on validation.

In [2]:
specs = M.model_specs(SEED)
tuning, chosen = [], {}
for name, (builder, param_grid) in specs.items():
    table, best = M.tune(name, builder, param_grid, train[M.FEATURES], train[M.TARGET], val[M.FEATURES], val[M.TARGET])
    tuning.append(table)
    model = builder(best, train[M.TARGET]).fit(train[M.FEATURES], train[M.TARGET])
    prob_val = model.predict_proba(val[M.FEATURES])[:, 1]
    chosen[name] = {"params": best, "threshold": M.best_f1_threshold(val[M.TARGET], prob_val), "prob_val": prob_val}
    print(f"{name:20s} best {best}  val PR-AUC {table.loc[0, 'val_pr_auc']:.3f}  threshold {chosen[name]['threshold']:.3f}"
          f"  ({len(table)} settings tried)")
tuning = pd.concat(tuning, ignore_index=True)
tuning["params"] = tuning["params"].astype(str)
tuning.to_csv(config.PROCESSED_DIR / "tuning_results.csv", index=False)

Logistic regression  best {'C': 1}  val PR-AUC 0.849  threshold 0.770  (4 settings tried)


Decision tree        best {'max_depth': 4, 'min_samples_leaf': 10}  val PR-AUC 0.860  threshold 0.933  (12 settings tried)


Random forest        best {'max_depth': 8, 'min_samples_leaf': 1}  val PR-AUC 0.917  threshold 0.494  (12 settings tried)


XGBoost              best {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.03}  val PR-AUC 0.902  threshold 0.721  (12 settings tried)


## Step 2: Refit on train + validation, evaluate on test once

Baselines use no training: **current ratio < 1** (the rule in the project brief) and **persistence** (Z' < 1.23 this year predicts distress next year). Z' is never a model input; persistence is shown only to measure what the ratios add over knowing today's score. For ROC/PR-AUC the baselines rank rows by −current ratio and −Z'.

In [3]:
results, test_pred = [], test[["cik", "name", "label_year", M.TARGET, "distressed"]].copy()
final_models = {}
for name, (builder, _) in specs.items():
    c = chosen[name]
    val_metrics = M.evaluate(val[M.TARGET], c["prob_val"], (c["prob_val"] >= c["threshold"]).astype(int))
    model = builder(c["params"], train_val[M.TARGET]).fit(train_val[M.FEATURES], train_val[M.TARGET])
    final_models[name] = model
    prob = model.predict_proba(test[M.FEATURES])[:, 1]
    test_pred[f"prob_{name}"] = prob
    results.append({"model": name, "set": "validation", **val_metrics})
    results.append({"model": name, "set": "test", "threshold": c["threshold"], "params": str(c["params"]),
                    **M.evaluate(test[M.TARGET], prob, (prob >= c["threshold"]).astype(int))})

# Baselines (score = higher means more distressed). Rows without a current
# ratio or a Z' at t get the training median, so they count as "not flagged".
cr_fill = train["current_ratio"].median()
z_fill = train["z_prime"].median()
print("Rows without Z' at t (persistence filled with training median):",
      {"validation": int(val["z_prime"].isna().sum()), "test": int(test["z_prime"].isna().sum())})
for s_name, d in [("validation", val), ("test", test)]:
    cr = d["current_ratio"].fillna(cr_fill)
    results.append({"model": "Baseline: current ratio < 1", "set": s_name, **M.evaluate(d[M.TARGET], -cr, (cr < 1).astype(int))})
    z = d["z_prime"].fillna(z_fill)
    results.append({"model": "Baseline: persistence (Z' < 1.23 now)", "set": s_name,
                    **M.evaluate(d[M.TARGET], -z, (z < 1.23).astype(int))})
test_pred["score_current_ratio"] = -test["current_ratio"].fillna(cr_fill)
test_pred["score_persistence"] = -test["z_prime"].fillna(z_fill)

comparison = pd.DataFrame(results)
comparison.to_csv(config.PROCESSED_DIR / "model_comparison.csv", index=False)
test_pred.to_csv(config.PROCESSED_DIR / "test_predictions.csv", index=False)
cols = ["model", "precision", "recall", "f1", "roc_auc", "pr_auc", "tp", "fp", "fn", "tn"]
comparison[comparison["set"] == "test"][cols].sort_values("pr_auc", ascending=False)

Rows without Z' at t (persistence filled with training median): {'validation': 1, 'test': 0}


,model,precision,recall,f1,roc_auc,pr_auc,tp,fp,fn,tn
11,Baseline: persistence (Z' < 1.23 now),0.880,0.815,0.846,0.953,0.939,44,6,10,74
5,Random forest,0.746,0.815,0.779,0.895,0.883,44,15,10,65
7,XGBoost,0.812,0.722,0.765,0.894,0.878,39,9,15,71
1,Logistic regression,0.826,0.704,0.760,0.885,0.844,38,8,16,72
3,Decision tree,0.796,0.722,0.757,0.852,0.756,39,10,15,70
10,Baseline: current ratio < 1,1.000,0.074,0.138,0.756,0.715,4,0,50,80


In [4]:
# Validation vs test for each model: a test score far above validation would suggest leakage.
vt = comparison.pivot_table(index="model", columns="set", values=["f1", "roc_auc", "pr_auc"])
vt.columns = [f"{m}_{s}" for m, s in vt.columns]
vt["pr_auc_test_minus_val"] = vt["pr_auc_test"] - vt["pr_auc_validation"]
vt.sort_values("pr_auc_test", ascending=False)

,f1_test,f1_validation,pr_auc_test,pr_auc_validation,roc_auc_test,roc_auc_validation,pr_auc_test_minus_val
model,,,,,,,
Baseline: persistence (Z' < 1.23 now),0.846,0.847,0.939,0.910,0.953,0.926,0.028
Random forest,0.779,0.835,0.883,0.917,0.895,0.942,-0.034
XGBoost,0.765,0.795,0.878,0.902,0.894,0.942,-0.024
Logistic regression,0.760,0.843,0.844,0.849,0.885,0.920,-0.005
Decision tree,0.757,0.811,0.756,0.860,0.852,0.913,-0.104
Baseline: current ratio < 1,0.138,0.128,0.715,0.579,0.756,0.700,0.137


## Step 3: Confusion matrices on test

In [5]:
for _, r in comparison[comparison["set"] == "test"].iterrows():
    cm = pd.DataFrame([[r["tn"], r["fp"]], [r["fn"], r["tp"]]],
                      index=["actual healthy", "actual distressed"], columns=["predicted healthy", "predicted distressed"])
    print(f"=== {r['model']}")
    print(cm.to_string())
    print()

=== Logistic regression
                   predicted healthy  predicted distressed
actual healthy                    72                     8
actual distressed                 16                    38

=== Decision tree
                   predicted healthy  predicted distressed
actual healthy                    70                    10
actual distressed                 15                    39

=== Random forest
                   predicted healthy  predicted distressed
actual healthy                    65                    15
actual distressed                 10                    44

=== XGBoost
                   predicted healthy  predicted distressed
actual healthy                    71                     9
actual distressed                 15                    39

=== Baseline: current ratio < 1
                   predicted healthy  predicted distressed
actual healthy                    80                     0
actual distressed                 50                     4

=== Ba

## Step 4: Early-warning cases

Most distressed companies were already distressed the year before. The genuine early-warning cases are companies **healthy by Z' at t that became distressed at t+1**. Recall on these shows whether a model sees trouble coming before the score itself does (persistence catches none of them by construction).

In [6]:
new = test_pred[(test_pred["distressed"] == 0) & (test_pred[M.TARGET] == 1)]
print(f"Test rows healthy at t and distressed at t+1: {len(new)}")
ew = {}
for name in specs:
    caught = (new[f"prob_{name}"] >= chosen[name]["threshold"]).sum()
    ew[name] = {"caught": int(caught), "of": len(new), "recall": caught / len(new) if len(new) else np.nan}
ew["Baseline: current ratio < 1"] = {"caught": int((new["score_current_ratio"] > -1).sum()), "of": len(new),
                                     "recall": (new["score_current_ratio"] > -1).mean()}
ew["Baseline: persistence (Z' < 1.23 now)"] = {"caught": 0, "of": len(new), "recall": 0.0}
display(pd.DataFrame(ew).T)
new[["name", "label_year"] + [f"prob_{n}" for n in specs]]

Test rows healthy at t and distressed at t+1: 10


,caught,of,recall
Logistic regression,2.000,10.000,0.200
Decision tree,1.000,10.000,0.100
Random forest,3.000,10.000,0.300
XGBoost,3.000,10.000,0.300
Baseline: current ratio < 1,0.000,10.000,0.000
Baseline: persistence (Z' < 1.23 now),0.000,10.000,0.000


,name,label_year,prob_Logistic regression,prob_Decision tree,prob_Random forest,prob_XGBoost
223,MICROCHIP TECHNOLOGY INC,2023,0.419,0.041,0.119,0.042
476,"NLIGHT, INC.",2023,0.175,0.830,0.286,0.379
487,GSI TECHNOLOGY INC,2023,0.984,0.830,0.818,0.835
585,"MAXLINEAR, INC",2023,0.821,0.979,0.927,0.892
637,NXP SEMICONDUCTORS N.V.,2024,0.192,0.041,0.126,0.041
649,"SOLAREDGE TECHNOLOGIES, INC.",2023,0.240,0.412,0.372,0.090
662,EVERSPIN TECHNOLOGIES INC,2023,0.167,0.428,0.166,0.180
681,"ENPHASE ENERGY, INC.",2023,0.315,0.041,0.144,0.088
763,BROADCOM INC.,2023,0.172,0.412,0.161,0.084
813,"TIGO ENERGY, INC.",2023,0.690,0.173,0.688,0.786


## Step 5: Best model and feature importance

The best model is chosen by **validation** PR-AUC (choosing on test would let test results influence the choice). Importance is measured two ways on the test years: permutation importance (drop in PR-AUC when one feature's values are shuffled) and SHAP values (average contribution of each feature to the predicted probability).

In [7]:
val_rank = comparison[(comparison["set"] == "validation") & ~comparison["model"].str.startswith("Baseline")]
best_name = val_rank.sort_values("pr_auc", ascending=False)["model"].iloc[0]
best_model = final_models[best_name]
print("Best model by validation PR-AUC:", best_name, chosen[best_name]["params"])

perm = permutation_importance(best_model, test[M.FEATURES], test[M.TARGET], scoring="average_precision",
                              n_repeats=30, random_state=SEED)
importance = pd.DataFrame({"feature": M.FEATURES, "perm_importance_mean": perm.importances_mean,
                           "perm_importance_std": perm.importances_std})

Best model by validation PR-AUC: Random forest {'max_depth': 8, 'min_samples_leaf': 1}


In [8]:
# SHAP on the preprocessed test matrix (tree models: TreeExplainer).
X_test_t = best_model[:-1].transform(test[M.FEATURES])
explainer = shap.TreeExplainer(best_model[-1])
sv = explainer.shap_values(X_test_t)
sv = sv[..., 1] if np.ndim(sv) == 3 else (sv[1] if isinstance(sv, list) else sv)
shap_mean = pd.Series(np.abs(sv).mean(axis=0), index=X_test_t.columns)
# Fold "was missing" indicator columns into their ratio for a per-feature view.
shap_by_feature = shap_mean.groupby(lambda c: c.replace("missingindicator_", "")).sum()
importance["shap_mean_abs"] = importance["feature"].map(shap_by_feature).fillna(0)
importance["shap_share"] = importance["shap_mean_abs"] / importance["shap_mean_abs"].sum()
importance = importance.sort_values("perm_importance_mean", ascending=False).reset_index(drop=True)
importance.insert(0, "model", best_name)
importance.to_csv(config.PROCESSED_DIR / "feature_importance.csv", index=False)
importance

,model,feature,perm_importance_mean,perm_importance_std,shap_mean_abs,shap_share
0,Random forest,operating_margin,0.022,0.019,0.067,0.133
1,Random forest,debt_to_equity,0.022,0.010,0.041,0.083
2,Random forest,debt_ratio,0.019,0.010,0.040,0.079
3,Random forest,log_total_assets,0.018,0.012,0.040,0.079
4,Random forest,net_margin,0.016,0.018,0.060,0.120
5,Random forest,roa,0.014,0.017,0.056,0.111
6,Random forest,interest_coverage,0.008,0.011,0.067,0.135
7,Random forest,gross_margin,0.003,0.004,0.011,0.022
8,Random forest,quick_ratio,0.001,0.004,0.017,0.034
9,Random forest,no_interest_expense,0.001,0.002,0.007,0.014


In [9]:
# Leakage audit: does one feature dominate?
top_share = importance["shap_share"].max()
print(f"Largest single-feature share of mean |SHAP|: {top_share:.1%} ({importance.loc[importance['shap_share'].idxmax(), 'feature']})")
print(f"Top three features' share: {importance.nlargest(3, 'shap_share')['shap_share'].sum():.1%}")

# Logistic regression coefficients (scaled features): sign shows the direction of risk.
lr = final_models["Logistic regression"]
coef = pd.Series(lr[-1].coef_[0], index=lr[:-1].get_feature_names_out()).sort_values()
print()
print("Logistic regression coefficients (standardised; positive = raises distress risk):")
coef.to_frame("coefficient")

Largest single-feature share of mean |SHAP|: 13.5% (interest_coverage)
Top three features' share: 38.8%

Logistic regression coefficients (standardised; positive = raises distress risk):


,coefficient
roa,-1.569
operating_margin,-1.512
current_ratio,-1.268
log_total_assets,-0.928
interest_coverage,-0.806
coverage_uses_proxy,-0.717
no_interest_expense,-0.713
missingindicator_cash_ratio,-0.480
debt_to_equity,-0.383
missingindicator_debt_to_equity,-0.380


## Step 6: Case-company probabilities (walk-forward, out of sample)

For each feature year t from 2018 to 2025, the best model (same settings and threshold) is trained only on feature years up to t−1, whose labels were known by year t, and then scores every company in year t. Every probability is therefore out of sample; the 2025 row is the forecast for 2026. Settings were chosen on the 2021-2022 validation years, so early years carry a small look-ahead in the choice of settings (not in the data).

In [10]:
builder = specs[best_name][0]
wf = M.walk_forward(df, builder, chosen[best_name]["params"], chosen[best_name]["threshold"],
                    config.WALK_FORWARD_START, config.LABEL_YEAR_END)
wf["company"] = wf["cik"].map(config.CASE_COMPANIES)
wf.to_csv(config.PROCESSED_DIR / "walk_forward_predictions.csv", index=False)
print(f"Walk-forward predictions: {len(wf):,} company-years, {wf['label_year'].min()}-{wf['label_year'].max()}")

lab_wf = wf[wf[M.TARGET].notna()]
print("All companies, labelled years:", {k: round(v, 3) if isinstance(v, float) else v for k, v in
      M.evaluate(lab_wf[M.TARGET], lab_wf["prob_distressed_t1"], lab_wf["pred_distressed_t1"]).items()})

case_wf = wf[wf["company"].notna()]
prob = case_wf.pivot_table(index="label_year", columns="company", values="prob_distressed_t1")
actual = case_wf.pivot_table(index="label_year", columns="company", values=M.TARGET)
print("Predicted probability of distress in t+1 (rows = feature year t):")
display(prob)
print("Actual label in t+1 (blank for 2025: 2026 not yet known):")
display(actual)

Walk-forward predictions: 583 company-years, 2018-2025
All companies, labelled years: {'precision': 0.769, 'recall': 0.833, 'f1': 0.8, 'roc_auc': 0.922, 'pr_auc': 0.885, 'tp': 140, 'fp': 42, 'fn': 28, 'tn': 259, 'rows': 469, 'distressed': 168}
Predicted probability of distress in t+1 (rows = feature year t):


company,AMD,Intel,Nvidia,Wolfspeed
label_year,,,,
2018,0.345,0.139,0.150,0.781
2019,0.321,0.168,0.096,0.761
2020,0.114,0.184,0.065,0.729
2021,0.072,0.131,0.124,0.942
2022,0.107,0.278,0.173,0.831
2023,0.142,0.667,0.114,0.879
2024,0.100,0.850,0.091,0.844
2025,0.055,0.733,0.092,0.885


Actual label in t+1 (blank for 2025: 2026 not yet known):


company,AMD,Intel,Nvidia,Wolfspeed
label_year,,,,
2018,1.000,0.000,0.000,0.000
2019,0.000,0.000,0.000,1.000
2020,0.000,0.000,0.000,1.000
2021,0.000,0.000,0.000,1.000
2022,0.000,1.000,0.000,1.000
2023,0.000,1.000,0.000,1.000
2024,0.000,1.000,0.000,1.000


## Step 7: Robustness checks

1. **Robustness label** (net loss, coverage < 1.5 and current ratio < 1 in t+1) with the best model's settings: very few positives, so indicative only.
2. **Low-revenue companies included** in training and test, same settings.

In [11]:
checks = []
# 1. Robustness label
rt = "robust_label_t1"
rtrain, rval, rtest = (d[d[rt].notna()] for d in (train, val, test))
print("Robustness-label positives: train", int(rtrain[rt].sum()), "| val", int(rval[rt].sum()), "| test", int(rtest[rt].sum()))
m1 = builder(chosen[best_name]["params"], rtrain[rt]).fit(rtrain[M.FEATURES], rtrain[rt])
thr_r = M.best_f1_threshold(rval[rt], m1.predict_proba(rval[M.FEATURES])[:, 1]) if rval[rt].sum() > 0 else 0.5
rtv = pd.concat([rtrain, rval])
m1 = builder(chosen[best_name]["params"], rtv[rt]).fit(rtv[M.FEATURES], rtv[rt])
p = m1.predict_proba(rtest[M.FEATURES])[:, 1]
checks.append({"check": "Robustness label (test)", **M.evaluate(rtest[rt], p, (p >= thr_r).astype(int))})

# 2. Low-revenue companies included (same settings; threshold re-chosen on validation)
full = M.prepare(mt, include_low_revenue=True)
fm = M.split(full, config.TRAIN_YEARS, config.VAL_YEARS, config.TEST_YEARS)
ftr, fva, fte = (full[fm[k]] for k in ("train", "val", "test"))
m2 = builder(chosen[best_name]["params"], ftr[M.TARGET]).fit(ftr[M.FEATURES], ftr[M.TARGET])
thr_f = M.best_f1_threshold(fva[M.TARGET], m2.predict_proba(fva[M.FEATURES])[:, 1])
ftv = pd.concat([ftr, fva])
m2 = builder(chosen[best_name]["params"], ftv[M.TARGET]).fit(ftv[M.FEATURES], ftv[M.TARGET])
p2 = m2.predict_proba(fte[M.FEATURES])[:, 1]
checks.append({"check": "Low revenue included (all test rows)", **M.evaluate(fte[M.TARGET], p2, (p2 >= thr_f).astype(int))})
op = ~fte["low_revenue"].astype(bool)
checks.append({"check": "Low revenue included (test, operating companies only)",
               **M.evaluate(fte.loc[op, M.TARGET], p2[op.values], (p2[op.values] >= thr_f).astype(int))})
main = comparison[(comparison["model"] == best_name) & (comparison["set"] == "test")].iloc[0]
checks.append({"check": f"Main model ({best_name}, test)", **main[["precision", "recall", "f1", "roc_auc", "pr_auc", "tp", "fp", "fn", "tn", "rows", "distressed"]].to_dict()})
checks = pd.DataFrame(checks)
checks.to_csv(config.PROCESSED_DIR / "robustness_checks.csv", index=False)
checks

Robustness-label positives: train 5 | val 4 | test 2


,check,precision,recall,f1,roc_auc,pr_auc,tp,fp,fn,tn,rows,distressed
0,Robustness label (test),0.091,0.500,0.154,0.869,0.196,1,10,1,120,132,2
1,Low revenue included (all test rows),0.814,0.826,0.820,0.907,0.899,57,13,12,68,150,69
2,"Low revenue included (test, operating companie...",0.778,0.778,0.778,0.895,0.885,42,12,12,68,134,54
3,"Main model (Random forest, test)",0.746,0.815,0.779,0.895,0.883,44,15,10,65,134,54
